# Notebook 01 — Exploratory Data Analysis (EDA)
**Project:** isiZulu Authorship Attribution  
**Sprint 1, Day 1**

This notebook documents everything we know about the dataset *before* any modelling.  
It answers three questions:
1. How is the data distributed across authors?
2. How long are the articles?
3. What does the raw isiZulu text actually look like?

> **Rule:** This notebook is read-only. We never modify the raw CSV here.

## 1. Imports and setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Make src importable from notebooks/
sys.path.insert(0, os.path.join("..", "src"))

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
FIGURES_DIR = os.path.join("..", "outputs", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print("✓ Imports OK")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")

## 2. Load the raw dataset

In [ ]:
RAW_PATH = os.path.join("..", "data", "raw",
                    "isizulu_authors_dataset_cleaned.csv")

df = pd.read_csv(RAW_PATH)
print(f"Shape  : {df.shape}  ({df.shape[0]} articles, {df.shape[1]} columns)")
print(f"Columns: {df.columns.tolist()}")
print()
df.head(3)

## 3. Data quality checks

In [ ]:
# Missing values
print("=== Missing values ===")
print(df.isnull().sum())

# Exact-duplicate articles
n_dup = df.duplicated(subset=["text"]).sum()
print(f"\n=== Exact duplicate articles: {n_dup} ===")

# Data types
print("\n=== dtypes ===")
print(df.dtypes)

### Expected results
- **0 missing values** in author, url, text columns  
- **0 exact duplicates** (each article is unique)  
- All columns are `object` (string) dtype — correct for text classification

## 4. Author distribution

In [ ]:
author_counts = df["author"].value_counts()
print("Author article counts:")
print(author_counts.to_string())
print(f"\nTotal articles : {len(df)}")
print(f"Unique authors : {df['author'].nunique()}")
print(f"\n⚠  Note: Simangaliso Ntshangase has only {author_counts.min()} articles.")
print("   This class imbalance will affect per-class F1 scores.")

In [ ]:
PALETTE = [
    "#4A90D9","#E8593C","#2ECC71","#F5A623",
    "#9B59B6","#1ABC9C","#E74C3C",
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
bars = axes[0].barh(author_counts.index, author_counts.values,
                    color=PALETTE, edgecolor="white")
axes[0].set_xlabel("Number of articles")
axes[0].set_title("Article count per author")
for bar, val in zip(bars, author_counts.values):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 str(val), va="center", fontsize=10)
axes[0].invert_yaxis()

# Pie chart
axes[1].pie(author_counts.values, labels=author_counts.index,
            colors=PALETTE, autopct="%1.1f%%",
            startangle=140, pctdistance=0.82,
            textprops={"fontsize": 9})
axes[1].set_title("Proportion of articles per author")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "author_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved: author_distribution.png")

## 5. Article length analysis

In [ ]:
df["char_len"]  = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

print("=== Character length statistics ===")
print(df["char_len"].describe().round(0).to_string())
print()
print("=== Word count statistics ===")
print(df["word_count"].describe().round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Character length histogram
axes[0].hist(df["char_len"], bins=25, color="#4A90D9",
             edgecolor="white", alpha=0.85)
axes[0].axvline(df["char_len"].mean(), color="#E8593C",
                linestyle="--", label=f"Mean = {df['char_len'].mean():.0f}")
axes[0].axvline(df["char_len"].median(), color="#2ECC71",
                linestyle=":", label=f"Median = {df['char_len'].median():.0f}")
axes[0].set_xlabel("Characters per article")
axes[0].set_ylabel("Count")
axes[0].set_title("Article length (characters)")
axes[0].legend()

# Word count histogram
axes[1].hist(df["word_count"], bins=25, color="#9B59B6",
             edgecolor="white", alpha=0.85)
axes[1].axvline(df["word_count"].mean(), color="#E8593C",
                linestyle="--", label=f"Mean = {df['word_count'].mean():.0f}")
axes[1].set_xlabel("Words per article")
axes[1].set_ylabel("Count")
axes[1].set_title("Article length (words)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "article_length_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved: article_length_distribution.png")

In [ ]:
# Per-author length breakdown
per_author = (df.groupby("author")["char_len"]
               .agg(["mean", "min", "max", "std", "count"])
               .round(0)
               .sort_values("mean", ascending=False))
per_author.columns = ["Mean chars", "Min chars", "Max chars", "Std chars", "Articles"]
print("=== Character length per author ===")
print(per_author.to_string())
print()
print("Key insight: Nonkululeko Nhlapo writes significantly shorter articles")
print("(mean ~1202 chars vs dataset mean ~2186). This stylometric signal")
print("may actually help the classifier identify her.")

In [ ]:
# Boxplot per author
fig, ax = plt.subplots(figsize=(12, 5))
order = df.groupby("author")["char_len"].median().sort_values(ascending=False).index
short_names = [n.split()[-1] for n in order]

bp = ax.boxplot(
    [df[df["author"] == a]["char_len"].values for a in order],
    labels=short_names,
    patch_artist=True,
    notch=False,
)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xlabel("Author (surname)")
ax.set_ylabel("Article length (characters)")
ax.set_title("Article length distribution per author")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "length_boxplot_per_author.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved: length_boxplot_per_author.png")

## 6. Sample text inspection

In [ ]:
print("=" * 70)
print("SAMPLE ARTICLE EXCERPTS (first 300 chars per author)")
print("=" * 70)

for author in df["author"].unique():
    sample_text = df[df["author"] == author]["text"].iloc[0]
    print(f"\nAuthor: {author}")
    print("-" * 40)
    print(sample_text[:300])
    print()

print("\nKey observations:")
print("• Text is authentic isiZulu — agglutinative prefixes visible (u-, ama-, isi-)")
print("• Proper nouns (IEC, KwaZulu-Natal) and code-switching (Image:) present")
print("• URLs and image captions are mixed in — may need cleaning in Sprint 2")

## 7. Vocabulary overlap between authors

In [ ]:
from collections import Counter

author_vocabs = {}
for author in df["author"].unique():
    texts = " ".join(df[df["author"] == author]["text"].tolist())
    words = set(texts.lower().split())
    author_vocabs[author] = words

all_authors = list(author_vocabs.keys())
print("Unique word counts per author:")
for a, vocab in author_vocabs.items():
    print(f"  {a:<30} {len(vocab):>5} unique tokens")

# Pairwise overlap (Jaccard similarity)
print("\nPairwise Jaccard vocabulary overlap (lower = more distinct writing style):")
for i, a1 in enumerate(all_authors):
    for a2 in all_authors[i+1:]:
        intersection = len(author_vocabs[a1] & author_vocabs[a2])
        union        = len(author_vocabs[a1] | author_vocabs[a2])
        jaccard      = intersection / union
        print(f"  {a1.split()[-1]:<15} vs {a2.split()[-1]:<22} Jaccard = {jaccard:.3f}")

## 8. EDA Summary and implications for modelling

In [ ]:
print("=" * 65)
print("EDA SUMMARY")
print("=" * 65)
print(f"  Total articles  : 157")
print(f"  Unique authors  : 7")
print(f"  Mean length     : ~2,186 characters / ~350 words")
print(f"  Class balance   : Mostly balanced (11–25 per author)")
print()
print("IMPLICATIONS:")
print("  1. USE STRATIFIED SPLIT — Simangaliso has only 11 articles.")
print("     Without stratification, he could be missing from the test set.")
print()
print("  2. USE F1 MACRO (not accuracy) as the primary metric.")
print("     Class imbalance means accuracy is misleading — a model that")
print("     ignores Simangaliso still gets ~93% accuracy by chance.")
print()
print("  3. SMALL VOCAB SIZE for BPE (2000–3000) is correct.")
print("     With only ~109 training articles, a large vocab memorises")
print("     whole words, defeating the purpose of subword tokenisation.")
print()
print("  4. NONKULULEKO NHLAPO writes shorter articles (~1202 chars mean).")
print("     This stylometric feature may be naturally learned by the model.")
print()
print("  5. SIMANGALISO NTSHANGASE is the hardest author to classify.")
print("     Expect his per-class F1 to be the lowest across all models.")
print("     Flag this in your evaluation write-up.")